# BR01A — Reproduce original PIL image preprocessing

The batch-only check did not resolve the discrepancy. This check explicitly selects the PIL processor backend, retains original unique-admission batching for Symile, and uses unchanged numerical tolerances. It checks the original five probes and then an expanded set of 20 per source. Images and labels used for encoder verification come from reference banks, not evaluation outcomes. Run in the RAD-DINO CUDA environment. Prior audits are preserved, and no frozen Python files are changed.


In [ ]:
from pathlib import Path
import sys
EXP = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'config.example.json').is_file() and (p / 'common.py').is_file()), None)
if EXP is None: raise FileNotFoundError('Start Jupyter inside graph_reasoning/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read
CONFIG_PATH = EXP / 'config.json'
if not CONFIG_PATH.is_file():
    raise FileNotFoundError('Run BR00 first to create experiment02/config.json')
CFG = load_config(CONFIG_PATH)
PROJECT = parent_setup(CFG)
RUN = Path(CFG['run_dir'])
print('Kernel:', sys.executable)
print('Cohort:', CFG['cohort'], '| outputs:', RUN)


In [ ]:
import hashlib
from unittest.mock import patch
import numpy as np
from PIL import Image
import encoders
from common import sha, atomic, immutable

SNAPSHOT_PATH = None  # Same original microsoft/rad-dino snapshot.
PROBES = 5
ORIGINAL_BATCH_SIZE = 16  # Default in notebook2/04_encode_cxr_raddino.ipynb.
# If the ORIGINAL encoding job overrode RADDINO_BATCH_SIZE, use that exact value.
if type(ORIGINAL_BATCH_SIZE) is not int or ORIGINAL_BATCH_SIZE < 1:
    raise ValueError('Original batch size must be a positive integer')

bank = read(RUN / 'banks/symile.json')
folder = Path(CFG['symile_root']) / 'data_npy/train'
raw = np.load(folder / 'cxr_train.npy', mmap_mode='r', allow_pickle=False)
hadm = np.load(folder / 'hadm_id_train.npy', mmap_mode='r', allow_pickle=False).reshape(-1)
if len(raw) != len(hadm): raise ValueError('CXR/admission array length mismatch')
# Original encoder processed UNIQUE admissions in np.unique's sorted order.
_, first_index, inverse = np.unique(hadm, return_index=True, return_inverse=True)
rng = np.random.default_rng(CFG['seed'])
probe_rows = [bank[i] for i in rng.choice(len(bank), min(PROBES, len(bank)), replace=False)]

def to_image(array):
    x = np.asarray(array, dtype=np.float32)
    x = x * np.array([.229, .224, .225], dtype=np.float32)[:, None, None]
    x = x + np.array([.485, .456, .406], dtype=np.float32)[:, None, None]
    return Image.fromarray((np.clip(x, 0, 1).transpose(1, 2, 0) * 255).round().astype('uint8'))

def pixel_hash(image):
    x = np.ascontiguousarray(np.asarray(image.convert('RGB')))
    return hashlib.sha256(x.tobytes()).hexdigest()

original_encode = encoders.rad_encode
state = {'calls': 0, 'batches': {}, 'details': []}

def encode_with_original_batches(bundle, image):
    number = state['calls']
    state['calls'] += 1
    # The verifier processes all Symile probes first, then MIMIC probes.
    # MIMIC already passed single-image checks and retains the same procedure.
    if number >= len(probe_rows):
        return original_encode(bundle, image)
    row = probe_rows[number]
    source_index = int(row['source_index'])
    if str(hadm[source_index]).removesuffix('.0') != str(row['admission_id']):
        raise ValueError('Probe admission mismatch')
    if pixel_hash(image) != pixel_hash(to_image(raw[source_index])):
        raise ValueError('Unexpected verifier probe order or image preprocessing')
    unique_position = int(inverse[source_index])
    # Original pipeline encoded position 0 alone as its first probe, then 1..N in batches.
    start = 0 if unique_position == 0 else 1 + ((unique_position - 1) // ORIGINAL_BATCH_SIZE) * ORIGINAL_BATCH_SIZE
    stop = 1 if start == 0 else min(start + ORIGINAL_BATCH_SIZE, len(first_index))
    key = (start, stop)
    if key not in state['batches']:
        images = [to_image(raw[int(i)]) for i in first_index[start:stop]]
        torch = bundle['torch']
        with torch.inference_mode():
            pixels = bundle['processor'](images=images, return_tensors='pt')['pixel_values']
            output = bundle['model'](pixel_values=pixels.to('cuda', dtype=torch.float16))
            pooled = output.pooler_output if getattr(output, 'pooler_output', None) is not None else output.last_hidden_state[:, 0]
        state['batches'][key] = pooled.float().cpu().numpy()
    state['details'].append({'id': row['id'], 'source_index': source_index,
                             'unique_admission_position': unique_position,
                             'batch_start': start, 'batch_stop': stop})
    return state['batches'][key][unique_position - start]

# Explicit PIL processing avoids an implicit torchvision/PIL backend change.
from transformers import AutoImageProcessor
ATTENTION_IMPLEMENTATION = None  # Preserve current attention; only change if original environment is documented.
original_bundle = encoders.rad_bundle
runtime_cache = {}

def bundle_with_original_processor(cfg, snapshot=None):
    if not runtime_cache:
        bundle = original_bundle(cfg, snapshot)
        old_processor = bundle['processor']
        processor = AutoImageProcessor.from_pretrained(
            str(bundle['snapshot']), local_files_only=True, backend='pil', use_fast=False)
        name = processor.__class__.__name__
        if 'fast' in name.lower() or 'torchvision' in processor.__class__.__module__.lower():
            raise ValueError('Explicit PIL request still returned a fast/torchvision processor')
        bundle['processor'] = processor
        if ATTENTION_IMPLEMENTATION is not None:
            setter = getattr(bundle['model'], 'set_attn_implementation', None)
            if setter is None: raise ValueError('Model cannot explicitly select the requested attention implementation')
            setter(ATTENTION_IMPLEMENTATION)
        state['runtime'] = {
            'previous_processor_class': old_processor.__class__.__name__,
            'previous_processor_module': old_processor.__class__.__module__,
            'processor_class': name,
            'processor_module': processor.__class__.__module__,
            'processor_config': processor.to_dict(),
            'attention_implementation': getattr(bundle['model'].config, '_attn_implementation', None),
            'cuda_device': bundle['torch'].cuda.get_device_name(),
            'dtype': str(next(bundle['model'].parameters()).dtype)
        }
        print('Processor/runtime:', state['runtime'])
        runtime_cache['bundle'] = bundle
    return runtime_cache['bundle']


In [ ]:
from common import signature
import traceback

audit_path = RUN / 'audits/rad_equivalence.json'
if audit_path.exists():
    previous = read(audit_path)
    immutable(RUN / 'audits/rad_history' / (signature(previous) + '.json'), previous)

all_checks = []
all_details = []
verification_error = None
expanded_complete = False
# First reproduce the original five probes; then check an expanded training-only set.
for number in (5, 20):
    probe_rows = [bank[i] for i in np.random.default_rng(CFG['seed']).choice(
        len(bank), min(number, len(bank)), replace=False)]
    state['calls'] = 0
    state['batches'] = {}
    state['details'] = []
    previous_stamp = audit_path.stat().st_mtime_ns if audit_path.exists() else None
    try:
        with patch.object(encoders, 'rad_bundle', side_effect=bundle_with_original_processor), \
             patch.object(encoders, 'rad_encode', side_effect=encode_with_original_batches):
            encoders.verify_rad(CFG, snapshot=SNAPSHOT_PATH, probes=number)
    except Exception as exc:
        verification_error = str(exc)
        print('Verification stopped:', type(exc).__name__, str(exc))
        traceback.print_exc(limit=2)
    # Never relabel an older audit if a failure occurred before verification wrote a new one.
    if not audit_path.exists() or audit_path.stat().st_mtime_ns == previous_stamp:
        diagnostic = {'passed': False, 'error': verification_error or 'No new verification audit',
                      'runtime': state.get('runtime'), 'requested_probes_per_source': number}
        atomic(RUN / 'audits/rad_processor_diagnostic.json', diagnostic)
        raise RuntimeError('No new equivalence audit was produced; inspect rad_processor_diagnostic.json')
    current = read(audit_path)
    all_checks.extend(current['checks'])
    all_details.extend(dict(item, requested_probes=number) for item in state['details'])
    print('Requested probes per source:', number)
    for check in current['checks']: print(check)
    if not current['passed']: break
    if number == 20: expanded_complete = True

result = dict(current)
result['checks'] = all_checks
result['passed'] = expanded_complete and all(check['passed'] for check in all_checks)
result['verification_procedure'] = {
    'name': 'explicit_pil_original_symile_batches_expanded_recheck',
    'original_batch_size': ORIGINAL_BATCH_SIZE,
    'symile_probe_batches': all_details,
    'runtime': state.get('runtime'),
    'expanded_check_complete': expanded_complete,
    'thresholds_unchanged': {'max_relative_l2': 0.005, 'min_cosine': 0.99999},
    'unique_probes': len({(c['source'], c['id']) for c in all_checks}),
    'recheck_notebook_sha256': sha(EXP / 'BR01A_original_processor_rad_recheck.ipynb'),
    'error': verification_error,
    'note': 'Explicit original-style PIL preprocessing; no stored embeddings, weights or frozen Python files changed.'
}
atomic(audit_path, result)
print('Verification passed:', result['passed'])
print('Audit:', audit_path)
if not result['passed']:
    raise ValueError('Original-processor recheck failed. Inspect audit; do not override passed or loosen thresholds.')
